# WARAS — fitting parameter squat (Chen et al. 2026)
Ini bukan training deep learning. Notebook ini cuma mencari angka untuk aturan di `lib/squat.ts`:
- `k, b`: regresi linear `sudut_panggul = k * sudut_lutut + b` dari squat standar
- `tol`: `min(3*sigma, 15)` derajat
- `delta`: toleransi kedalaman (panggul boleh di atas lutut sejauh `delta * panjang_paha`), persentil 95 squat standar
- `minPassRatio`: porsi frame minimal yang harus masuk pita `tol`

Akurasi diukur dengan **leave-one-subject-out**: tiap subjek diuji pakai parameter yang di-fit tanpa subjek itu.

Output: `waras_params.json` → taruh di `lib/waras_params.json` di repo, push, Vercel deploy ulang otomatis.

**Langkah:** Runtime → Run all. Di sel upload, pilih ZIP dari Zenodo (link ada di bagian *Data Availability* artikel, doi:10.3390/computers15050293). Boleh upload semua ZIP sekaligus; file yang sama otomatis dihitung sekali.

In [ ]:
import numpy as np, pandas as pd, json, glob, os, zipfile
from google.colab import files

## 1. Upload data Zenodo (CSV atau ZIP berisi CSV)

In [ ]:
up = files.upload()
os.makedirs('data', exist_ok=True)
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall(f'data/{name[:-4]}')
    else:
        os.replace(name, f'data/{name}')
# 02_Err_GoodMorning.zip juga berisi semua klip lain, jadi dedupe per nama file
paths = sorted({os.path.basename(p): p for p in glob.glob('data/**/*.csv', recursive=True)}.values())
print(len(paths), 'klip unik')

## 2. Baca klip
Format Zenodo: satu CSV per repetisi, kolom MediaPipe `x_i, y_i, z_i, v_i`, nama file `pXX_<Jenis>_NN.csv`
(Jenis: `Standard`, `Shallow`, `GM`). Label diambil dari **nama file**, bukan folder.

Sisi badan (kiri/kanan) dipilih per frame berdasarkan visibility, sama persis dengan `jointAngles()` di web.
`ASPECT` = lebar/tinggi video kalau koordinat dinormalisasi 0–1 (MediaPipe); isi 1 kalau koordinat dalam piksel.

In [ ]:
ASPECT = 1.0
LEFT, RIGHT = (11, 23, 25, 27), (12, 24, 26, 28)  # bahu, panggul, lutut, pergelangan kaki

def load_clip(path):
    d = pd.read_csv(path)
    vis = lambda ids: d[[f'v_{i}' for i in ids]].mean(axis=1).values
    use_left = vis(LEFT) >= vis(RIGHT)
    out = pd.DataFrame({f'{p}{c}': np.where(use_left, d[f'{c}_{l}'], d[f'{c}_{r}'])
                        for p, l, r in zip('shka', LEFT, RIGHT) for c in 'xy'})
    name = os.path.basename(path)
    out['clip'] = name
    out['subject'] = name.split('_')[0]
    out['kind'] = name.split('_')[1].lower()          # standard / shallow / gm
    out['label'] = int(out['kind'][0] == 'standard')  # 1 = standar
    return out

df = pd.concat([load_clip(p) for p in paths], ignore_index=True)
print(df.groupby('kind').clip.nunique(), '\nsubjek:', df.subject.nunique())

## 3. Sudut sendi (hukum cosinus), sama persis dengan `angle()` di `lib/squat.ts`

In [ ]:
def angle(ax_, ay_, bx, by, cx, cy):
    ab = np.hypot(ax_ - bx, ay_ - by); cb = np.hypot(cx - bx, cy - by); ac = np.hypot(ax_ - cx, ay_ - cy)
    cos = np.clip((ab**2 + cb**2 - ac**2) / (2 * ab * cb), -1, 1)
    return np.degrees(np.arccos(cos))

X = lambda c: df[c] * ASPECT
df['hip']  = angle(X('sx'), df.sy, X('hx'), df.hy, X('kx'), df.ky)
df['knee'] = angle(X('hx'), df.hy, X('kx'), df.ky, X('ax'), df.ay)
df['thigh'] = np.hypot(X('hx') - X('kx'), df.hy - df.ky)
df = df.dropna(subset=['hip', 'knee', 'thigh'])
df[['hip', 'knee', 'thigh']].describe()

## 4. Fitting + aturan evaluasi (logika sama dengan `evaluateRep()` di web)

In [ ]:
def depth_gap(c):  # (lutut_y - panggul_y) / paha di frame panggul paling bawah; y gambar membesar ke bawah
    r = c.loc[c.hy.idxmax()]
    return (r.ky - r.hy) / r.thigh

def fit(data):
    std = data[data.label == 1]
    k, b = np.polyfit(std.knee, std.hip, 1)
    tol = float(min(3 * np.std(std.hip - (k * std.knee + b)), 15))
    ratios = [(np.abs(c.hip - (k * c.knee + b)) <= tol).mean() for _, c in std.groupby('clip')]
    delta = float(max(0.0, np.percentile([depth_gap(c) for _, c in std.groupby('clip')], 95)))
    return dict(k=float(k), b=float(b), tol=tol, delta=delta, minPassRatio=float(min(0.9, np.percentile(ratios, 5))))

def is_standard(c, p):
    coord_ok = (np.abs(c.hip - (p['k'] * c.knee + p['b'])) <= p['tol']).mean() >= p['minPassRatio']
    return bool(coord_ok and depth_gap(c) <= p['delta'])

## 5. Validasi leave-one-subject-out
- **Regresi** (per frame, squat standar subjek yang diuji): MAE, MSE, RMSE, R², MAPE dalam derajat.
- **Klasifikasi** (per klip): kelas positif = squat **salah** (yang harus ditegur).

In [ ]:
preds, resid = [], []
for s in sorted(df.subject.unique()):
    p = fit(df[df.subject != s]); te = df[df.subject == s]
    preds += [(kind, lab, is_standard(c, p)) for (_, kind, lab), c in te.groupby(['clip', 'kind', 'label'])]
    st = te[te.label == 1]
    resid.append(st.hip - (p['k'] * st.knee + p['b']))

e = pd.concat(resid); y = df.loc[e.index, 'hip']
print('REGRESI  MAE  = %.2f°' % e.abs().mean())
print('         MSE  = %.2f' % (e**2).mean())
print('         RMSE = %.2f°' % np.sqrt((e**2).mean()))
print('         R²   = %.4f' % (1 - (e**2).sum() / ((y - y.mean())**2).sum()))
print('         MAPE = %.2f%%' % ((e.abs() / y).mean() * 100))

res = pd.DataFrame(preds, columns=['kind', 'label', 'pred'])
yt, yp = res.label == 0, ~res.pred
tp, fp, fn, tn = (yt & yp).sum(), (~yt & yp).sum(), (yt & ~yp).sum(), (~yt & ~yp).sum()
prec, rec, spec = tp / (tp + fp), tp / (tp + fn), tn / (tn + fp)
print('\nKLASIFIKASI  akurasi     = %.2f%%  (paper 91.67%%)' % ((yt == yp).mean() * 100))
print('             precision   = %.2f%%' % (prec * 100))
print('             recall      = %.2f%%' % (rec * 100))
print('             specificity = %.2f%%' % (spec * 100))
print('             F1          = %.2f%%' % (200 * prec * rec / (prec + rec)))
print('\nbenar per jenis:', res.assign(ok=yt == yp).groupby('kind').ok.mean().round(3).to_dict())
pd.crosstab(res.label, res.pred, rownames=['label (1=standar)'], colnames=['prediksi standar'])

## 6. Fit final pakai semua subjek, simpan & download `waras_params.json`

In [ ]:
p = fit(df)
params = {'source': 'colab/waras_fit.ipynb on Zenodo data', 'k': round(p['k'], 4), 'b': round(p['b'], 4),
          'tol': round(p['tol'], 2), 'delta': round(p['delta'], 3), 'minPassRatio': round(p['minPassRatio'], 2)}
json.dump(params, open('waras_params.json', 'w'), indent=2)
print(params)
files.download('waras_params.json')